# Model Training and Evaluation

This notebook prepares the cleaned dataset for machine learning, encodes the categorical features, trains a Random Forest Regressor, tunes its main hyperparameters, and evaluates the final model.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder, TargetEncoder


## Load the cleaned dataset

The data used here has already gone through the cleaning steps from the preprocessing notebook, including duplicate removal, type conversion, and outlier handling.


In [ ]:
data = pd.read_csv('../data/car_price_prediction_cleaned.csv')

display(data.columns)


## Train/test split

The target is `Price`, while `Doors` is excluded from the feature set because it is not used in the final model.

The split is performed before fitting the encoders so that information from the test set does not influence the training process.


In [ ]:
# Features and target
X = data.drop(columns=["Price", "Doors"])
y = data["Price"]

# Keeping a separate test set for the final evaluation
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)


## Encoding categorical features

Different categorical variables require different encoding strategies.

- One-hot encoding is used for lower-cardinality features because it represents each category without imposing an artificial order.
- Target encoding is used for `Manufacturer` and `Model`, which contain many categories. This keeps the number of resulting features much smaller than one-hot encoding.
- Binary features are mapped directly to 0 and 1.

For all encoders, the training data is fitted first and the test data is only transformed. This prevents information from the test set from leaking into the training process.


In [ ]:
one_hot_features = [
    "Category",
    "Fuel type",
    "Gear box type",
    "Drive wheels",
    "Color"
]

target_features = [
    "Manufacturer",
    "Model"
]

binary_features = [
    "Leather interior",
    "Wheel"
]

# One-hot encoding for categorical features with a manageable number of categories
one_hot_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

X_train_one_hot = one_hot_encoder.fit_transform(X_train[one_hot_features])
X_test_one_hot = one_hot_encoder.transform(X_test[one_hot_features])

# Target encoding for high-cardinality categorical features
target_encoder = TargetEncoder(
    cv=5,
    target_type="continuous"
)

X_train_target = target_encoder.fit_transform(
    X_train[target_features],
    y_train
)

X_test_target = target_encoder.transform(
    X_test[target_features]
)

# Binary encoding for features with two meaningful categories
X_train_binary = X_train[binary_features].copy()
X_test_binary = X_test[binary_features].copy()

X_train_binary["Leather interior"] = X_train_binary["Leather interior"].map({
    "Yes": 1,
    "No": 0
})

X_test_binary["Leather interior"] = X_test_binary["Leather interior"].map({
    "Yes": 1,
    "No": 0
})

X_train_binary["Wheel"] = X_train_binary["Wheel"].map({
    "Right-hand drive": 1,
    "Left wheel": 0
})

X_test_binary["Wheel"] = X_test_binary["Wheel"].map({
    "Right-hand drive": 1,
    "Left wheel": 0
})


## Build the final feature matrices

`OneHotEncoder` and `TargetEncoder` return NumPy arrays, so they are converted back to DataFrames. This allows us to keep meaningful feature names and align the encoded data with the original training and test indices.

The numerical and encoded categorical features are then combined into the final matrices that will be passed to the model. The original categorical columns are not included again because their encoded versions already represent them.


In [ ]:
one_hot_columns = one_hot_encoder.get_feature_names_out(one_hot_features)

X_train_one_hot = pd.DataFrame(
    X_train_one_hot,
    columns=one_hot_columns,
    index=X_train.index
)

X_test_one_hot = pd.DataFrame(
    X_test_one_hot,
    columns=one_hot_columns,
    index=X_test.index
)

X_train_target = pd.DataFrame(
    X_train_target,
    columns=target_features,
    index=X_train.index
)

X_test_target = pd.DataFrame(
    X_test_target,
    columns=target_features,
    index=X_test.index
)

numerical_features = [
    "Levy",
    "Prod. year",
    "Engine volume",
    "Mileage",
    "Cylinders"
]

X_train_numeric = X_train[numerical_features].copy()
X_test_numeric = X_test[numerical_features].copy()

# Combine all prepared features into the final training and test sets
X_train_final = pd.concat(
    [
        X_train_numeric,
        X_train_target,
        X_train_binary,
        X_train_one_hot
    ],
    axis=1
)

X_test_final = pd.concat(
    [
        X_test_numeric,
        X_test_target,
        X_test_binary,
        X_test_one_hot
    ],
    axis=1
)

print(X_train_final.shape)
print(X_test_final.shape)
display(X_train_final.head())

print("Missing values in X_train_final:", X_train_final.isna().sum().sum())
print("Missing values in X_test_final:", X_test_final.isna().sum().sum())


## Random Forest hyperparameter tuning

The Random Forest was chosen because it can capture non-linear relationships and interactions between vehicle characteristics without requiring the same assumptions as a linear model.

I initially considered a wider hyperparameter search with more cross-validation folds. However, this made the tuning process considerably more expensive on this dataset. I therefore reduced the search space to the main parameters affecting the forest and used 3-fold cross-validation with 10 sampled combinations.

This gives a practical balance between exploring the model and keeping the computation time reasonable.


In [ ]:
hyper_params = {
    "n_estimators": [100, 200, 300],
    "max_depth": [10, 20, 30, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": [0.5, 0.75, 1.0]
}

model = RandomForestRegressor(
    random_state=42,
    n_jobs=1
)

random_search = RandomizedSearchCV(
    estimator=model,
    param_distributions=hyper_params,
    n_iter=10,
    cv=3,
    scoring="neg_mean_absolute_error",
    random_state=42,
    n_jobs=-1,
    verbose=1
)

random_search.fit(X_train_final, y_train)


## Evaluate the tuned model

The test set is used only after tuning to obtain an unbiased final evaluation.

For this regression problem:

- **MAE** gives the average absolute prediction error in the same unit as the car price.
- **RMSE** penalizes larger errors more strongly than MAE.
- **R²** measures how much of the variation in the target is explained by the model.


In [ ]:
best_model = random_search.best_estimator_

y_pred = best_model.predict(X_test_final)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f"RMSE: {rmse:.2f}")
print(f"R2: {r2:.4f}")
print(f"MAE: {mae:.2f}")

print("Best parameters:")
print(random_search.best_params_)

print(f"Best cross-validation MAE: {-random_search.best_score_:.2f}")


## Feature importance

Random Forest provides an importance value for each feature based on its contribution to reducing impurity across the trees.

The importance values should be interpreted as a measure of the model's reliance on the features, not as a direct percentage change in car price.


In [ ]:
importance_df = pd.DataFrame({
    "Feature": X_train_final.columns,
    "Importance": best_model.feature_importances_
})

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

display(importance_df)


## Final model

After tuning, the selected hyperparameters are used to train the final Random Forest model. Keeping this step separate makes the final model configuration explicit and easy to reproduce.


In [ ]:
final_model = RandomForestRegressor(
    random_state=42,
    n_estimators=150,
    max_depth=20,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features=0.75
)

final_model.fit(X_train_final, y_train)

y_pred = final_model.predict(X_test_final)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f"RMSE: {rmse:.2f}")
print(f"R2: {r2:.4f}")
print(f"MAE: {mae:.2f}")
